In [1]:
import os
from dotenv import load_dotenv

load_dotenv()


True

In [11]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'

print(device)

################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
# from ssonn.metrics.train_metrics import *
from ssonn.train.train import *


cpu


In [12]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=16):
        super(SimpleFCN, self).__init__()
        # для инварианта положительности, пока нет другого фикса
        self.fc_input = nn.Linear(input_size, hidden_size)
        self.act = nn.ReLU()
        self.fc_output = nn.Linear(hidden_size, 10)


    def forward(self, x):
        x = self.fc_input(x)
        x = self.act(x)
        x = self.fc_output(x)
        return x


In [13]:
hyperparams = {
    "num_epochs": 32,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc_output": 0.4},
    "pruning_thresholds": {"fc_output": 1},
    "plateau_threshold": 0.05,
    "min_epochs_between_expansions": 8,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 2000,
    "weight_decay": 1e-4,
    "diff_threshold": 0.05,
    "warmup_epochs": 5  # количество эпох на восстановление lr
}

In [14]:
# Dataset and Dataloader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

# Load dataset and split into train/validation sets
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=g)

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = int(finder_fraction * len(train_dataset))
finder_dataset = Subset(train_dataset.dataset, train_dataset.indices[:n_finder])

batch_size = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
finder_loader = DataLoader(finder_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [15]:
torch.manual_seed(SEED)
model = SimpleFCN()
model = convert_to_scaled_net(model, layers_to_convert=[model.fc_output], device=device)

Converting layer 'fc_output' to ScaledNet...


In [7]:
# import wandb
# wandb.login()

In [8]:
# wandb.finish()
# run = wandb.init(
#     project="self-expanding-nets",
#     name=f"name",
#     config=hyperparams
# )

In [9]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="mnist",
)
experiment.log_parameters(hyperparams)
experiment.set_name("mnist origin")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gcof/mnist/a270ee8c2c654f7fab607ce98fc43242

COMET WARNING: Unknown error exporting current conda environment
COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


In [16]:
criterion = nn.CrossEntropyLoss()

def optimizer_factory(m):
    return optim.Adam(m.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])

trained_model = train_ssonn(
    model,
    train_loader,
    val_loader,
    finder_loader,
    criterion,
    optimizer_factory,
    hyperparams,
    device,
    experiment,
    True,
)

--- Starting SSONN Training ---


Epoch 1/32 | T-Loss: 1.0321 | V-Loss: 0.4806 | Acc: 0.8796 | Params: 12730/12730


Epoch 2/32 | T-Loss: 0.3963 | V-Loss: 0.3493 | Acc: 0.9059 | Params: 12730/12730


Epoch 3/32 | T-Loss: 0.3246 | V-Loss: 0.3110 | Acc: 0.9139 | Params: 12730/12730


Epoch 4/32 | T-Loss: 0.2951 | V-Loss: 0.2921 | Acc: 0.9196 | Params: 12730/12730


Epoch 5/32 | T-Loss: 0.2761 | V-Loss: 0.2797 | Acc: 0.9228 | Params: 12730/12730


... (23 outputs left out)

Epoch 29/32 | T-Loss: 0.1460 | V-Loss: 0.1843 | Acc: 0.9471 | Params: 13458/13486


Epoch 30/32 | T-Loss: 0.1439 | V-Loss: 0.1821 | Acc: 0.9472 | Params: 13458/13486


Epoch 31/32 | T-Loss: 0.1423 | V-Loss: 0.1833 | Acc: 0.9484 | Params: 13458/13486


Epoch 32/32 | T-Loss: 0.1405 | V-Loss: 0.1816 | Acc: 0.9481 | Params: 13458/13486
--- Training Finished ---


In [17]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : mnist origin
COMET INFO:     url                   : https://www.comet.com/gcof/mnist/a270ee8c2c654f7fab607ce98fc43242
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     expansion event     : 28
COMET INFO:     lr [64]             : (0.0005, 0.001)
COMET INFO:     params amount [64]  : (12730, 13458)
COMET INFO:     sparsity [64]       : (0.0, 0.004152454397152616)
COMET INFO:     total params [64]   : (12730, 13486)
COMET INFO:     train accuracy [64] : (0.7415208333333333, 0.959375)
COMET INFO:     train loss [64]     : (0.1405004909896153, 1.03212429123356)
COMET INFO:     val accuracy [64]   : (0.8795833333333334, 0.9484166666666667)
COMET

In [43]:
torch.save(model, 'mnist.pt')